In [9]:
import requests
import pandas as pd


def getDataFromKegg(operation, argument):
    url = f"https://rest.kegg.jp/{operation}/{argument}"

    resp = requests.get(
        url
    )

    if resp.ok:
        return resp.text
    
with open('uniprot_id', 'r') as plik:
    dane = plik.read()
    lista_uniprot_id = dane.split('\n')
    print(lista_uniprot_id)


['P10321', 'P04439', 'P01889', 'P17693', 'P13747']


In [11]:
lista_kegg_id = []

for element in lista_uniprot_id:
    kegg_id = getDataFromKegg('conv', f'genes/uniprot:{element}').split('\t')
    kegg_id = kegg_id[1].strip('\n')
    lista_kegg_id.append(kegg_id)

print(lista_kegg_id)

['hsa:3107', 'hsa:3105', 'hsa:3106', 'hsa:3135', 'hsa:3133']


In [13]:
with open('aaseq', 'w') as plik:
    for element in lista_kegg_id:
        plik.write(getDataFromKegg('get', f'{element}/aaseq'))

In [16]:
import time

def runClustal(sequecnes_in_fasta):
    resp = requests.post(
        'https://www.ebi.ac.uk/Tools/services/rest/clustalo/run',
        data={
            "email": "marcel.thiel@ug.edu.pl",
            "sequence": sequecnes_in_fasta,
            "outfmt": "fa"
        }
    )
    jobid = resp.text

    return jobid


def getStatus(jobId):
    url = f"https://www.ebi.ac.uk/Tools/services/rest/clustalo/status/{jobId}"
    
    resp = requests.get(
        url
    )

    return resp.text


def getResults(jobId):
    url = f"https://www.ebi.ac.uk/Tools/services/rest/clustalo/result/{jobId}/fa"

    resp = requests.get(
        url
    )

    return resp.text

with open('aaseq', 'r') as plik:
    data = plik.read()
jobid = runClustal(data)

for i in range(30):
    if getStatus(jobid) == 'FINISHED':
        msaResults = getResults(jobid)
        print(msaResults)
        break
    else:
        time.sleep(1)


>3107 K06751 MHC class I antigen | (RefSeq) HLA-C, D6S204, HLA-JY3, HLAC, HLC-C, MHC, PSORS1; major histocompatibility complex, class
MRVMAPRALLLLLSGGLALTETWACSHSMRYFDTAVSRPGRGEPRFISVGYVDDTQFVRF
DSDAASPRGEPRAPWVEQEGPEYWDRETQKYKRQAQADRVSLRNLRGYYNQSEDGSHTLQ
RMSGCDLGPDGRLLRGYDQSAYDGKDYIALNEDLRSWTAADTAAQITQRKLEAARAAEQL
RAYLEGTCVEWLRRYLENGKETLQRAEPPKTHVTHHPLSDHEATLRCWALGFYPAEITLT
WQRDGEDQTQDTELVETRPAGDGTFQKWAAVVVPSGQEQRYTCHMQHEGLQEPLTLSWEP
SSQPTIPIMGIVAGLAVLVVLAVLGAVVTAMMCRRKSSGGKGGSCSQAACSNSAQGSDES
LITCKA
>3105 K06751 MHC class I antigen | (RefSeq) HLA-A, HLAA; major histocompatibility complex, class I, A (A)
MAVMAPRTLLLLLSGALALTQTWAGSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRF
DSDAASQKMEPRAPWIEQEGPEYWDQETRNMKAHSQTDRANLGTLRGYYNQSEDGSHTIQ
IMYGCDVGPDGRFLRGYRQDAYDGKDYIALNEDLRSWTAADMAAQITKRKWEAVHAAEQR
RVYLEGRCVDGLRRYLENGKETLQRTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLT
WQRDGEDQTQDTELVETRPAGDGTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRWEL
SSQPTIPIVGIIAGLVLLGA-VITGAVVAAVMWRRKSSDRKGGSYTQAASSDSAQGSDVS
LTACKV
>3106 K06751 M

In [17]:
linijki = msaResults.split('\n')
ilosc_gap = 0
for linijka in linijki:
    if linijka.startswith('>'):
        pass
    else:
        ilosc_gap += linijka.count('-')
print(ilosc_gap)

41
